# H2S-NN Node Classification W&B Tuning - Colab

Primary W&B tuning notebook for ACM/DBLP/IMDB node classification. Defaults target AMPSS configs, train on the complete graph in one optimizer step per epoch, and use isolated, fingerprinted trials for Colab time-limit restarts. Query minibatching is intentionally not part of this initial sweep.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)
print('Python executable:', sys.executable)


In [ ]:
import importlib.metadata
import os
import signal
import subprocess
import sys

requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Missing pinned requirements: {requirements_path}')

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''

target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    '-r',
    str(requirements_path),
])

if not target_ready:
    print('Pinned Torch/CUDA stack installed. Restarting the runtime; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')


In [ ]:
import subprocess
import sys

import torch
import torch_geometric
import torch_scatter
import torch_sparse

if not torch.__version__.startswith('2.6.'):
    raise RuntimeError(f'Expected torch 2.6.*, found {torch.__version__}. Rerun the install cell.')
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 Torch build, found CUDA {torch.version.cuda}.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('torch_scatter:', torch_scatter.__version__)
print('torch_sparse:', torch_sparse.__version__)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name)
    print('GPU capacity GiB:', round(properties.total_memory / 1024**3, 2))

subprocess.run([sys.executable, 'scripts/run_h2snn_wandb_tuning.py', '--help'], check=True)


In [ ]:
from pathlib import Path

CONFIG_PATH = Path('configs/experiments/acm_h2snn_ampss.yaml')  # acm/dblp/imdb *_ampss.yaml
EXP_NAME = 'acm_h2snn_ampss_node_fullbatch_primary_v5'
OUTPUT_DIR = Path('experiments/colab')
STATE_DIR = OUTPUT_DIR / EXP_NAME / 'tuning_state'

NUM_RUNS = 5
N_TRIALS = 50
EPOCHS = 500
PATIENCE = 50
HYPERBAND_MIN_ITER = 30
HYPERBAND_ETA = 3
D_VALUES = [8,16, 32]
BASE_SEED = 42
RUN_START = 0
RUN_COUNT = NUM_RUNS

RESUME = False
SWEEP_ID = ''
WANDB_PROJECT = 'h2snn'
WANDB_ENTITY = ''
WANDB_MODE = 'online'
WANDB_TAGS = ['paper-quality', 'node-classification', 'wandb-tuning']

print('After tuning:', OUTPUT_DIR / EXP_NAME / 'tuning_summary.json')
print('Repeated-run summaries use runs_summary.json from scripts/run_h2snn_runs.py')


import yaml

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Base config does not exist: {CONFIG_PATH}')
base_payload = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
base_orthogonal_map = str(base_payload.get('model', {}).get('orthogonal_map', 'fasthpp'))
if base_orthogonal_map != 'fasthpp':
    raise ValueError('The current src model and tuning protocol support only orthogonal_map=fasthpp.')
print('Resolved base orthogonal map:', base_orthogonal_map)
print('Requested d values:', D_VALUES or 'safe task defaults')
print('Training context: complete graph; query minibatching is not enabled')
print('Fresh sweep:', not RESUME)


In [ ]:
import os
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()


In [ ]:
import subprocess
import sys

cmd = [sys.executable, 'scripts/run_h2snn_wandb_tuning.py', '--config', str(CONFIG_PATH), '--exp_name', EXP_NAME, '--output_dir', str(OUTPUT_DIR), '--device', DEVICE, '--num_runs', str(NUM_RUNS), '--n_trials', str(N_TRIALS), '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--hyperband_min_iter', str(HYPERBAND_MIN_ITER), '--hyperband_eta', str(HYPERBAND_ETA), '--state_dir', str(STATE_DIR), '--wandb_project', WANDB_PROJECT, '--wandb_mode', WANDB_MODE, '--wandb_tags', *WANDB_TAGS]
if D_VALUES:
    cmd.extend(['--d_values', *[str(value) for value in D_VALUES]])
cmd.extend(['--base_seed', str(BASE_SEED), '--run_start', str(RUN_START), '--run_count', str(RUN_COUNT)])
if WANDB_ENTITY:
    cmd.extend(['--wandb_entity', WANDB_ENTITY])
if RESUME:
    cmd.append('--resume')
if SWEEP_ID:
    cmd.extend(['--sweep_id', SWEEP_ID])
print('Validating:', ' '.join([*cmd, '--validate_only']))
subprocess.run([*cmd, '--validate_only'], check=True)
print('Starting:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
summary_path = OUTPUT_DIR / EXP_NAME / 'tuning_summary.json'
print('summary_path:', summary_path)
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding='utf-8')).get('summary', {}), indent=2)[:4000])
else:
    for path in sorted(STATE_DIR.glob('tuning_state_run*.json')):
        state = json.loads(path.read_text(encoding='utf-8'))
        print(path.name, 'completed', state.get('completed_trials'), '/', state.get('target_trials'), 'sweep', state.get('sweep_id'))
